In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
from urllib.parse import quote_plus

load_dotenv()

host = os.getenv("TIDB_HOST")
port = os.getenv("TIDB_PORT", "4000")
user = os.getenv("TIDB_USER")
password = os.getenv("TIDB_PASSWORD")
database = os.getenv("TIDB_DATABASE")

engine = create_engine(
    f"mysql+pymysql://{quote_plus(user)}:{quote_plus(password)}@{host}:{port}/{database}",
    connect_args={"ssl": {}}
)

with engine.connect() as conn:
    print("Connected successfully!")
    print(conn.execute(text("SHOW TABLES")).fetchall())

Connected successfully!
[('cast_clean',), ('crew_clean',), ('genres',), ('movie_genres',), ('movie_genres_clean',), ('movie_keywords_clean',), ('movies',)]


# Reel Insights TMDB — SQL Analysis

## Objective
Analyze the TMDB movie database using SQL to identify trends in revenue, ratings, popularity, languages, genres, and movie releases.

In [2]:
from sqlalchemy import text
import pandas as pd

with engine.connect() as conn:
    tables = pd.read_sql(text("SHOW TABLES"), conn)

display(tables)

,Tables_in_reel_insights_tmdb
0,cast_clean
1,crew_clean
2,genres
3,movie_genres
4,movie_genres_clean
5,movie_keywords_clean
6,movies


In [3]:
with engine.connect() as conn:
    for table in [
        "movies",
        "genres",
        "movie_genres",
        "cast_clean",
        "crew_clean",
        "movie_keywords_clean"
    ]:
        print(f"\n--- {table} ---")
        result = conn.execute(text(f"DESCRIBE {table}"))
        for row in result.fetchall():
            print(row)


--- movies ---
('movie_id', 'int', 'NO', 'PRI', None, '')
('title', 'varchar(500)', 'YES', '', None, '')
('original_language', 'varchar(10)', 'YES', '', None, '')
('release_date', 'date', 'YES', '', None, '')
('runtime', 'int', 'YES', '', None, '')
('budget', 'bigint', 'YES', '', None, '')
('revenue', 'bigint', 'YES', '', None, '')
('popularity', 'double', 'YES', '', None, '')
('vote_average', 'double', 'YES', '', None, '')
('vote_count', 'int', 'YES', '', None, '')
('status', 'varchar(50)', 'YES', '', None, '')
('overview', 'text', 'YES', '', None, '')

--- genres ---
('genre_id', 'int', 'YES', '', None, '')
('genre_name', 'text', 'YES', '', None, '')

--- movie_genres ---
('movie_id', 'int', 'YES', '', None, '')
('genre_id', 'int', 'YES', '', None, '')

--- cast_clean ---
('cast_row_id', 'int', 'YES', '', None, '')
('movie_id', 'int', 'YES', '', None, '')
('person_id', 'int', 'YES', '', None, '')
('actor_name', 'text', 'YES', '', None, '')
('character_name', 'text', 'YES', '', None,

In [4]:
with engine.connect() as conn:
    for table in [
        "cast_clean",
        "crew_clean",
        "movie_keywords_clean"
    ]:
        result = conn.execute(text(f"DESCRIBE {table}"))
        print(f"\n{table}:")
        print([row[0] for row in result.fetchall()])


cast_clean:
['cast_row_id', 'movie_id', 'person_id', 'actor_name', 'character_name', 'cast_order']

crew_clean:
['crew_row_id', 'movie_id', 'person_id', 'person_name', 'job', 'department']

movie_keywords_clean:
['movie_id', 'keyword_id', 'keyword_name']


## Question 1: Action Movies Released After 2015

List all Action movies released after 2015, including their title and release date.

In [5]:
query = """
SELECT DISTINCT
    m.title,
    m.release_date
FROM movies m
JOIN movie_genres mg
    ON m.movie_id = mg.movie_id
JOIN genres g
    ON mg.genre_id = g.genre_id
WHERE g.genre_name = 'Action'
  AND m.release_date >= '2016-01-01'
ORDER BY m.release_date;
"""

with engine.connect() as conn:
    result = pd.read_sql(text(query), conn)

display(result)

print(f"Total Action movies released after 2015: {len(result)}")

,title,release_date
0,The 5th Wave,2016-01-14
1,Kung Fu Panda 3,2016-01-23
2,Deadpool,2016-02-09
3,Gods of Egypt,2016-02-25
4,London Has Fallen,2016-03-02
...,...,...
199,Jurassic World Rebirth,2025-06-23
200,F1,2025-06-25
201,Superman,2025-07-09
202,The Fantastic 4: First Steps,2025-07-23


Total Action movies released after 2015: 204


### Insight
This analysis identifies Action movies released after 2015 and shows their release dates, helping us understand recent releases in the Action genre.

## Question 2: Top 10 Highest-Grossing Movies

List the top 10 highest-grossing movies along with their genres.

In [6]:
query = """
SELECT
    m.title,
    m.revenue,
    GROUP_CONCAT(DISTINCT g.genre_name ORDER BY g.genre_name SEPARATOR ', ') AS genres
FROM movies m
LEFT JOIN movie_genres mg
    ON m.movie_id = mg.movie_id
LEFT JOIN genres g
    ON mg.genre_id = g.genre_id
WHERE m.revenue IS NOT NULL
  AND m.revenue > 0
GROUP BY m.movie_id, m.title, m.revenue
ORDER BY m.revenue DESC
LIMIT 10;
"""

with engine.connect() as conn:
    result = pd.read_sql(text(query), conn)

display(result)

,title,revenue,genres
0,Avatar,2923706026,"Action, Adventure, Science Fiction"
1,Avengers: Endgame,2799439100,"Action, Adventure, Science Fiction"
2,Avatar: The Way of Water,2334484620,"Action, Adventure, Science Fiction"
3,Titanic,2264162353,"Drama, Romance"
4,Star Wars: The Force Awakens,2068223624,"Action, Adventure, Science Fiction"
5,Avengers: Infinity War,2052415039,"Action, Adventure, Science Fiction"
6,Spider-Man: No Way Home,1921206586,"Action, Adventure, Science Fiction"
7,Zootopia 2,1868208796,NaN
8,Inside Out 2,1698863816,"Adventure, Animation, Comedy, Family"
9,Jurassic World,1671537444,"Adventure, Science Fiction, Thriller"


### Insight
This analysis highlights the 10 highest-grossing movies and their genres, showing which films generated the most revenue.

## Question 3: Average Budget and Revenue by Genre

Calculate the average budget and average revenue for each genre, sorted by average revenue in descending order.

In [7]:
query = """
SELECT
    g.genre_name,
    AVG(NULLIF(m.budget, 0)) AS average_budget,
    AVG(NULLIF(m.revenue, 0)) AS average_revenue
FROM movies m
JOIN movie_genres mg
    ON m.movie_id = mg.movie_id
JOIN genres g
    ON mg.genre_id = g.genre_id
GROUP BY g.genre_id, g.genre_name
ORDER BY average_revenue DESC;
"""

with engine.connect() as conn:
    result = pd.read_sql(text(query), conn)

display(result)

,genre_name,average_budget,average_revenue
0,Adventure,1.235223e+08,4.702789e+08
1,Animation,9.360103e+07,4.268124e+08
2,Family,9.434678e+07,3.990600e+08
3,Science Fiction,1.069178e+08,3.787745e+08
4,Action,1.061699e+08,3.656547e+08
5,Fantasy,9.816595e+07,3.582373e+08
6,Music,4.978519e+07,2.801893e+08
7,Comedy,6.194133e+07,2.674062e+08
8,Romance,3.972831e+07,2.114973e+08
9,Thriller,5.685453e+07,2.091203e+08


### Insight
Comparing average budget and revenue across genres helps identify genres that generate higher average revenue and require larger investments.

## Question 4: Top 10 Actors by Number of Movies

Identify the top 10 actors who have appeared in the highest number of movies.

In [8]:
query = """
SELECT
    actor_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM cast_clean
WHERE actor_name IS NOT NULL
  AND actor_name <> ''
GROUP BY person_id, actor_name
ORDER BY movie_count DESC, actor_name
LIMIT 10;
"""

with engine.connect() as conn:
    result = pd.read_sql(text(query), conn)

display(result)

,actor_name,movie_count
0,Samuel L. Jackson,39
1,Brad Pitt,38
2,Robert De Niro,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Mark Wahlberg,32
6,Scarlett Johansson,32
7,Willem Dafoe,32
8,Morgan Freeman,31
9,Tom Cruise,31


### Insight
This analysis identifies the 10 most prolific actors based on the number of distinct movies they appeared in.


## Question 5: Directors with More Than 3 Movies

Find directors who have directed more than 3 movies and display their total movie count.


In [9]:
query = """
SELECT
    person_name AS director_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM crew_clean
WHERE job = 'Director'
  AND person_name IS NOT NULL
  AND person_name <> ''
GROUP BY person_id, person_name
HAVING COUNT(DISTINCT movie_id) > 3
ORDER BY movie_count DESC, director_name;
"""

with engine.connect() as conn:
    result = pd.read_sql(text(query), conn)

display(result)

,director_name,movie_count
0,Steven Spielberg,27
1,Ridley Scott,19
2,Tim Burton,19
3,Martin Scorsese,16
4,Michael Bay,15
...,...,...
212,Stephen Sommers,4
213,Tate Taylor,4
214,Tim Johnson,4
215,Tim Story,4


### Insight
This analysis reveals directors with more than three movies, highlighting those with a larger number of directing credits in the dataset.

## Question 6: Top 10 Most Frequently Used Keywords

Identify the 10 most frequently used keywords across all movies.

In [10]:
query = """
SELECT
    keyword_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM movie_keywords_clean
WHERE keyword_name IS NOT NULL
  AND keyword_name <> ''
GROUP BY keyword_id, keyword_name
ORDER BY movie_count DESC, keyword_name
LIMIT 10;
"""

with engine.connect() as conn:
    result = pd.read_sql(text(query), conn)

display(result)

,keyword_name,movie_count
0,based on novel or book,449
1,sequel,416
2,duringcreditsstinger,296
3,aftercreditsstinger,242
4,murder,176
5,based on true story,166
6,new york city,161
7,villain,160
8,3d animation,155
9,based on comic,154


### Insight
The most frequently used keywords reveal recurring themes and topics represented across the movie collection.

## Question 7: Movies with Above-Average Budget but Below-Average Revenue

Find movies whose budget is above the overall average budget but whose revenue is below the overall average revenue.

In [11]:
query = """
SELECT
    title,
    budget,
    revenue
FROM movies
WHERE budget > (
    SELECT AVG(budget)
    FROM movies
    WHERE budget > 0
)
AND revenue < (
    SELECT AVG(revenue)
    FROM movies
    WHERE revenue > 0
)
AND budget > 0
AND revenue > 0
ORDER BY budget DESC;
"""

with engine.connect() as conn:
    result = pd.read_sql(text(query), conn)

display(result)

,title,budget,revenue
0,The Marvels,274800000,206136825
1,Red One,250000000,185700759
2,Wake Up Dead Man: A Knives Out Mystery,210000000,4000000
3,The Gray Man,200000000,454023
4,Mulan,200000000,69965374
...,...,...,...
268,Rambo III,63000000,189015611
269,The Equalizer 2,62000000,190400157
270,Æon Flux,62000000,53321673
271,2 Guns,61000000,131940411


### Insight
These movies had budgets above the dataset average but revenues below the dataset average, helping identify potentially underperforming investments.

## Question 8: Actors in Movies Directed by Christopher Nolan

List the actors who appeared in movies directed by Christopher Nolan.

In [12]:
query = """
SELECT DISTINCT
    c.actor_name,
    m.title AS movie_title,
    d.person_name AS director_name
FROM cast_clean c
JOIN movies m
    ON c.movie_id = m.movie_id
JOIN crew_clean d
    ON m.movie_id = d.movie_id
WHERE d.job = 'Director'
  AND LOWER(d.person_name) = 'christopher nolan'
  AND c.actor_name IS NOT NULL
ORDER BY m.title, c.actor_name;
"""

with engine.connect() as conn:
    result = pd.read_sql(text(query), conn)

display(result)

,actor_name,movie_title,director_name
0,Christian Bale,Batman Begins,Christopher Nolan
1,Cillian Murphy,Batman Begins,Christopher Nolan
2,Gary Oldman,Batman Begins,Christopher Nolan
3,Katie Holmes,Batman Begins,Christopher Nolan
4,Ken Watanabe,Batman Begins,Christopher Nolan
...,...,...,...
105,Michael Caine,The Prestige,Christopher Nolan
106,Piper Perabo,The Prestige,Christopher Nolan
107,Rebecca Hall,The Prestige,Christopher Nolan
108,Samantha Mahurin,The Prestige,Christopher Nolan


### Insight
This analysis identifies actors who appeared in movies directed by Christopher Nolan and shows the associated movie titles.

## Question 9: Genre with the Highest Average Rating

Find the genre with the highest average movie rating, considering only genres with at least 20 movies.

In [13]:
query = """
SELECT
    g.genre_name,
    AVG(m.vote_average) AS average_rating,
    COUNT(DISTINCT m.movie_id) AS movie_count
FROM movies m
JOIN movie_genres mg
    ON m.movie_id = mg.movie_id
JOIN genres g
    ON mg.genre_id = g.genre_id
WHERE m.vote_average IS NOT NULL
GROUP BY g.genre_id, g.genre_name
HAVING COUNT(DISTINCT m.movie_id) >= 20
ORDER BY average_rating DESC
LIMIT 1;
"""

with engine.connect() as conn:
    result = pd.read_sql(text(query), conn)

display(result)

,genre_name,average_rating,movie_count
0,War,7.468932,59


### Insight
Comparing average ratings among genres with at least 20 movies helps identify the highest-rated genre while reducing the influence of genres with very few movies.

## Question 10: Top 10 Movies with the Highest Number of Keywords

Find the top 10 movies that have the highest number of associated keywords.

In [14]:
query = """
SELECT
    m.title,
    COUNT(DISTINCT k.keyword_id) AS keyword_count
FROM movies m
JOIN movie_keywords_clean k
    ON m.movie_id = k.movie_id
GROUP BY m.movie_id, m.title
ORDER BY keyword_count DESC, m.title
LIMIT 10;
"""

with engine.connect() as conn:
    result = pd.read_sql(text(query), conn)

display(result)

,title,keyword_count
0,Smile 2,98
1,Taken 3,70
2,Silent Hill,63
3,Twelve Monkeys,57
4,War for the Planet of the Apes,54
5,The Shining,52
6,Dawn of the Planet of the Apes,51
7,How to Train Your Dragon,50
8,13 Hours: The Secret Soldiers of Benghazi,49
9,How to Train Your Dragon: The Hidden World,49


### Insight
The top 10 movies with the most associated keywords reveal which films have the richest keyword metadata in the dataset.